# Esercitazione 1: Post-Processing delle Previsioni di Precipitazione

**Modulo 2 — Deep Learning: Best Practice e Applicazioni**  
Corso di Dottorato, Fondazione CIMA, 27 maggio 2026

In questo notebook costruiamo una semplice CNN per il post-processing (correzione del bias) delle previsioni di precipitazione NWP.  
Utilizziamo lo **stack ML moderno**: Zarr + xarray per i dati, PyTorch Lightning per il training, Hydra/OmegaConf per la configurazione e MLFlow per il tracciamento degli esperimenti.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meteocima/Monaco_DLCourse/blob/main/notebooks/01_precipitation_postprocessing_it.ipynb)


## 0. Setup

Usiamo [**uv**](https://docs.astral.sh/uv/) come gestore di pacchetti Python — un sostituto rapido e moderno di pip + virtualenv.

| Contesto | Comando | Cosa fa |
|----------|---------|--------|
| **Nuovo progetto** | `uv init` | Crea lo scaffold di un progetto con `pyproject.toml` |
| **Aggiungere una dipendenza** | `uv add torch xarray` | Aggiunge pacchetti a `pyproject.toml` e li installa |
| **Installare tutto** | `uv sync` | Crea un `.venv` e installa dal lockfile |
| **Eseguire un comando** | `uv run jupyter lab` | Esegue all'interno del `.venv` gestito |
| **Colab / Python di sistema** | `uv pip install --system -r requirements.txt` | Installa senza `.venv` (per ambienti non gestiti) |

Su **Colab** non c'è un `.venv` — Google fornisce l'ambiente Python — quindi usiamo `uv pip install --system`.


In [ ]:
import os

# Su Colab: clona il repo (o aggiorna all'ultima versione) e installa le dipendenze
if "COLAB_RELEASE_TAG" in os.environ:
    if os.path.exists("/content/Monaco_DLCourse"):
        !git -C /content/Monaco_DLCourse pull -q
    else:
        !git clone https://github.com/meteocima/Monaco_DLCourse.git /content/Monaco_DLCourse
    %cd /content/Monaco_DLCourse/notebooks
    !pip install -q uv
    !uv pip install --system -q -r ../requirements.txt


In [ ]:
from __future__ import annotations

from typing import Any

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from omegaconf import DictConfig, OmegaConf
import mlflow

print(f"Versione PyTorch: {torch.__version__}")
print(f"Versione Lightning: {pl.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "\n\u26a0\ufe0f  Nessuna GPU rilevata!\n"
        "   Vai su Runtime \u2192 Change runtime type \u2192 T4 GPU\n"
        "   Poi riesegui questa cella.\n"
        "   Il training funzioner\u00e0 anche su CPU, ma sar\u00e0 pi\u00f9 lento."
    )


## 1. Configurazione con Hydra / OmegaConf

In produzione, Hydra carica file di configurazione YAML e li compone automaticamente tramite il suo decoratore `@hydra.main`. In un notebook usiamo **OmegaConf** (il backend di configurazione di Hydra) per caricare direttamente gli stessi file YAML — ottenendo una configurazione strutturata e tipizzata senza bisogno della CLI.

La funzione `load_config()` qui sotto carica il file YAML e permette di **sovrascrivere qualsiasi valore** tramite argomenti keyword — lo stesso schema che Hydra usa sulla riga di comando (`max_epochs=50`), adattato per i notebook.


In [ ]:
def load_config(
    config_path: str = "../configs/precipitation.yaml",
    **overrides: Any,
) -> DictConfig:
    """Carica una configurazione YAML e applica le sovrascritture.

    Args:
        config_path: Percorso del file di configurazione YAML.
        **overrides: Chiavi abbreviate (es. ``max_epochs=50``,
            ``batch_size=64``) mappate automaticamente ai
            rispettivi percorsi annidati nella configurazione.

    Returns:
        ``DictConfig`` OmegaConf risultante dalla fusione.
    """
    cfg = OmegaConf.load(config_path)

    # Mappa le chiavi abbreviate ai percorsi annidati nella configurazione
    SHORTCUTS: dict[str, str] = {
        "max_lead_time": "data.max_lead_time",
        "max_epochs": "training.max_epochs",
        "hidden_channels": "model.hidden_channels",
        "n_layers": "model.n_layers",
        "learning_rate": "training.learning_rate",
        "batch_size": "data.batch_size",
        "time_start": "data.time_start",
        "time_end": "data.time_end",
        "test_start": "data.test_start",
        "test_end": "data.test_end",
        "train_split": "data.train_split",
    }
    for key, value in overrides.items():
        path = SHORTCUTS.get(key, key)
        OmegaConf.update(cfg, path, value)

    return cfg


cfg = load_config()
print(OmegaConf.to_yaml(cfg))


## 2. Caricamento dei dati con Zarr + xarray

La funzione `load_data()` racchiude l'intera pipeline dei dati:

1. **Apertura** degli store Zarr cloud di WeatherBench 2 (lazy — nessun dato scaricato)
2. **Selezione** della verità ERA5 e delle previsioni HRES a tutti i lead time a intervalli di 6 ore fino a `max_lead_time`
3. **Conversione** da metri a **mm** (× 1000) — ERA5 memorizza la precipitazione totale in metri equivalenti di acqua; moltiplicando per 1000 si ottengono millimetri, più intuitivi per le applicazioni meteorologiche
4. **Normalizzazione** tramite scaling min-max con percentili 1° / 99° (calcolati solo sullo split di training)

### Normalizzazione Min-Max

Riscaliamo ogni campo approssimativamente nell'intervallo [0, 1] usando:

$$x_{\text{norm}} = \frac{x - p_1}{p_{99} - p_1}$$

dove $p_1$ e $p_{99}$ sono il 1° e il 99° percentile dei dati di training. Perché i percentili invece del vero min/max? Le distribuzioni di precipitazione hanno outlier estremi (eventi convettivi intensi); usare il vero min/max comprimerebbe >99% dei valori in un intervallo minuscolo. I percentili offrono una distribuzione più utile, e valori occasionali fuori da [0, 1] sono accettabili.

> **Evitare il data leakage:** le statistiche di normalizzazione ($p_1$, $p_{99}$) sono calcolate **solo sullo split di training**. Se usassimo l'intero dataset, il modello "vedrebbe" implicitamente informazioni dai dati di validation/test attraverso lo scaling, gonfiando le stime di performance.

### Codifica del Lead Time

Il lead time della previsione (es. 6 h, 12 h, …, 48 h) è normalizzato a [0, 1] e **proiettato su un campo spaziale completo** della stessa dimensione della previsione. Questo fornisce alla CNN informazioni sia spaziali che temporali simultaneamente — ogni punto griglia "sa" quanto è distante la previsione nel tempo, permettendo al modello di apprendere pattern di correzione dipendenti dal lead time.

### Suddivisione basata sulle date

I campioni vengono suddivisi in train / validation / test per **data**, non casualmente. I campi meteorologici in giorni consecutivi sono fortemente correlati (autocorrelazione temporale), quindi una suddivisione casuale causerebbe il leakage di campioni quasi duplicati tra i set, rendendo i punteggi di validation eccessivamente ottimistici.


In [ ]:
def _validate_date_ranges(cfg: DictConfig) -> None:
    """Errore se il periodo di test si sovrappone al periodo train+val."""
    tv_start = pd.Timestamp(cfg.data.time_start)
    tv_end = pd.Timestamp(cfg.data.time_end)
    te_start = pd.Timestamp(cfg.data.test_start)
    te_end = pd.Timestamp(cfg.data.test_end)
    if tv_start <= te_end and te_start <= tv_end:
        raise ValueError(
            f"Test period [{te_start}, {te_end}] overlaps with "
            f"train+val period [{tv_start}, {tv_end}]. "
            "They must be disjoint."
        )


def load_data(cfg: DictConfig) -> dict[str, Any]:
    """Carica previsioni di precipitazione e verità da WeatherBench 2.

    Usa ``time_start``/``time_end`` per il periodo train+val e
    ``test_start``/``test_end`` per un periodo di test fisso. Il
    confine train/val è stabilito da ``train_split`` a livello di
    init-time.

    Args:
        cfg: Configurazione con chiavi ``data.*`` (URL, intervallo
            temporale, ``max_lead_time``, ``train_split``,
            ``test_start``, ``test_end``, ``lat_slice``).

    Returns:
        Dizionario con array normalizzati, array originali in mm,
        coordinate, statistiche di normalizzazione e metadati.
    """
    _validate_date_ranges(cfg)

    gcs_opts = {"token": "anon"}
    ds_era5 = xr.open_zarr(
        cfg.data.era5_zarr_url, storage_options=gcs_opts
    )
    ds_hres = xr.open_zarr(
        cfg.data.hres_zarr_url,
        storage_options=gcs_opts,
        decode_timedelta=True,
    )

    print("Variabili ERA5:", list(ds_era5.data_vars))
    print("Variabili HRES:", list(ds_hres.data_vars))
    print(
        f"\nLead time HRES: "
        f"{ds_hres.prediction_timedelta.values[[0, -1]]}"
    )

    max_lead = pd.Timedelta(cfg.data.max_lead_time)
    steps = pd.timedelta_range("6h", max_lead, freq="6h")
    n_steps = len(steps)
    print(f"Passi di lead time ({n_steps}): {steps[0]}  \u2026  {steps[-1]}")

    lat_sl = slice(0, cfg.data.lat_slice)

    # \u2500\u2500 Funzione ausiliaria per caricare un intervallo temporale \u2500\u2500\u2500\u2500\u2500
    def _load_range(t_start: str, t_end: str) -> tuple[
        np.ndarray, np.ndarray, np.ndarray, np.ndarray
    ]:
        fc_raw = (
            ds_hres[cfg.data.variable]
            .sel(prediction_timedelta=steps)
            .sel(time=slice(t_start, t_end))
            .isel(latitude=lat_sl)
            .transpose(
                "time", "prediction_timedelta", "latitude", "longitude"
            )
        )
        init_times = fc_raw.time.values
        valid_times_2d = init_times[:, None] + steps.values[None, :]
        truth_raw = (
            ds_era5[cfg.data.variable]
            .sel(time=valid_times_2d.ravel())
            .isel(latitude=lat_sl)
            .transpose("time", "latitude", "longitude")
        )
        lats = fc_raw.latitude.values
        lons = fc_raw.longitude.values
        fc_mm = (
            fc_raw.values.astype(np.float32).reshape(
                -1, len(lats), len(lons)
            )
            * 1000
        )
        tr_mm = truth_raw.values.astype(np.float32) * 1000
        return fc_mm, tr_mm, init_times, lats

    # \u2500\u2500 Caricamento intervalli train+val e test \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    tv_fc_mm, tv_tr_mm, tv_init, lats = _load_range(
        cfg.data.time_start, cfg.data.time_end
    )
    te_fc_mm, te_tr_mm, te_init, _ = _load_range(
        cfg.data.test_start, cfg.data.test_end
    )
    lons = (
        ds_hres[cfg.data.variable]
        .isel(latitude=lat_sl)
        .longitude.values
    )

    n_tv_init = len(tv_init)
    n_te_init = len(te_init)
    n_train_init = int(n_tv_init * cfg.data.train_split)
    if n_train_init == 0:
        raise ValueError(
            f"train_split={cfg.data.train_split} produce 0 init time "
            f"di training su {n_tv_init}. Aumentare l\'intervallo "
            "temporale o train_split."
        )
    if n_train_init == n_tv_init:
        raise ValueError(
            f"train_split={cfg.data.train_split} non lascia init time "
            f"per la validation su {n_tv_init}. Diminuire train_split."
        )

    n_train_samples = n_train_init * n_steps
    n_trainval_samples = n_tv_init * n_steps

    print(f"\nForma HRES train+val: {tv_fc_mm.shape}")
    print(f"Forma HRES test:      {te_fc_mm.shape}")

    # \u2500\u2500 Normalizzazione solo dai dati di training \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    train_all = np.concatenate(
        [tv_fc_mm[:n_train_samples], tv_tr_mm[:n_train_samples]]
    )
    precip_p1 = np.percentile(train_all, 1).astype(np.float32)
    precip_p99 = np.percentile(train_all, 99).astype(np.float32)
    precip_scale = precip_p99 - precip_p1
    print(
        f"Normalizzazione precipitazione: p1 = {precip_p1:.4f} mm, "
        f"p99 = {precip_p99:.4f} mm"
    )

    # \u2500\u2500 Concatenazione train+val e test \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    forecast_mm = np.concatenate([tv_fc_mm, te_fc_mm], axis=0)
    truth_mm = np.concatenate([tv_tr_mm, te_tr_mm], axis=0)

    forecast = (forecast_mm - precip_p1) / precip_scale
    truth = (truth_mm - precip_p1) / precip_scale

    # Lead-norm: ripetizione per intervallo, poi concatenazione
    step_hours = (steps.total_seconds() / 3600).values.astype(np.float32)
    max_hours = float(max_lead.total_seconds() / 3600)
    lead_norm_unit = (step_hours / max_hours).astype(np.float32)
    lead_norm = np.concatenate([
        np.tile(lead_norm_unit, n_tv_init),
        np.tile(lead_norm_unit, n_te_init),
    ])

    # \u2500\u2500 Riepilogo \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    print(f"\n\u2500\u2500 Suddivisione basata sulle date \u2500\u2500")
    print(
        f"Train:  {n_train_init} init time "
        f"({tv_init[0]}  \u2026  {tv_init[n_train_init - 1]})"
    )
    print(
        f"Val:    {n_tv_init - n_train_init} init time "
        f"({tv_init[n_train_init]}  \u2026  {tv_init[-1]})"
    )
    print(
        f"Test:   {n_te_init} init time "
        f"({te_init[0]}  \u2026  {te_init[-1]})"
    )
    print(
        f"Campioni \u2014 train: {n_train_samples}, "
        f"val: {n_trainval_samples - n_train_samples}, "
        f"test: {n_te_init * n_steps}"
    )
    print(
        f"Forecast: {forecast.shape}, Truth: {truth.shape}, "
        f"Lead norm: {lead_norm.shape}"
    )
    print(
        f"Bias medio (forecast \u2212 truth): "
        f"{(forecast_mm - truth_mm).mean():.4f} mm"
    )

    return {
        "forecast": forecast,
        "truth": truth,
        "lead_norm": lead_norm,
        "forecast_mm": forecast_mm,
        "truth_mm": truth_mm,
        "lats": lats,
        "lons": lons,
        "precip_p1": precip_p1,
        "precip_p99": precip_p99,
        "precip_scale": precip_scale,
        "n_train_samples": n_train_samples,
        "n_trainval_samples": n_trainval_samples,
        "n_steps": n_steps,
        "step_hours": step_hours,
        "max_hours": max_hours,
        "init_times": np.concatenate([tv_init, te_init]),
        "test_init_times": te_init,
    }


data = load_data(cfg)


### Visualizzazione dei dati grezzi

Prima del training è importante ispezionare i dati. `plot_raw_data()` mostra una singola tripletta previsione / verità / errore su mappa, più la curva **RMSE in funzione del lead time** per la prima data di emissione. Questo aiuta a identificare bias sistematici che la CNN dovrebbe imparare a correggere.


In [ ]:
def plot_raw_data(
    data: dict[str, Any],
    sample_idx: int = -1,
) -> None:
    """Visualizza una singola coppia previsione / verità e l'RMSE in funzione del lead time.

    Args:
        data: Dizionario restituito da ``load_data()``.
        sample_idx: Indice nei passi di lead time del primo tempo
            di inizializzazione. ``-1`` (default) seleziona l'ultimo.
    """
    forecast = data["forecast"]
    truth = data["truth"]
    lats = data["lats"]
    lons = data["lons"]
    step_hours = data["step_hours"]
    n_steps = data["n_steps"]
    init_times = data["init_times"]
    precip_p1 = data["precip_p1"]
    precip_scale = data["precip_scale"]

    def _to_mm(x: np.ndarray) -> np.ndarray:
        return x * precip_scale + precip_p1

    if sample_idx < 0:
        sample_idx = n_steps + sample_idx
    idx = sample_idx

    sample_lead_h = step_hours[idx % n_steps]
    issue_date = pd.Timestamp(init_times[0]).strftime("%Y-%m-%d %HZ")
    proj = ccrs.PlateCarree()
    lon2d, lat2d = np.meshgrid(lons, lats)

    fc_mm = _to_mm(forecast[idx])
    tr_mm = _to_mm(truth[idx])

    fig, axes = plt.subplots(
        1, 3, figsize=(18, 5), subplot_kw={"projection": proj}
    )
    titles = ["Previsione IFS HRES", "Verit\u00e0 ERA5", "Errore della previsione"]
    plot_data = [fc_mm, tr_mm, fc_mm - tr_mm]
    cmaps = ["Blues", "Blues", "RdBu_r"]
    vmin_max = [(0, None), (0, None), (None, None)]

    for ax, title, d, cmap, (vn, vx) in zip(
        axes, titles, plot_data, cmaps, vmin_max
    ):
        im = ax.pcolormesh(
            lon2d, lat2d, d, cmap=cmap, vmin=vn, vmax=vx,
            transform=proj,
        )
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle="--")
        ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
        ax.set_title(title)
        plt.colorbar(
            im, ax=ax, orientation="horizontal", pad=0.05, shrink=0.8
        )

    fig.suptitle(
        f"Precipitazione 6h \u2014 lead {sample_lead_h:.0f}h, "
        f"emessa il {issue_date} (mm)",
        fontsize=14, y=1.02,
    )
    plt.tight_layout()
    plt.show()

    # RMSE in funzione del lead time per la prima data di emissione
    rmse_per_step = np.array([
        np.sqrt(np.mean((_to_mm(forecast[i]) - _to_mm(truth[i])) ** 2))
        for i in range(n_steps)
    ])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(step_hours, rmse_per_step, "o-", color="tab:blue")
    ax.set_xlabel("Lead time (ore)")
    ax.set_ylabel("RMSE (mm)")
    ax.set_title(
        f"HRES vs ERA5 \u2014 RMSE per lead time (emessa il {issue_date})"
    )
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_raw_data(data)


## 3. PyTorch Dataset e DataModule

### Perché separare Dataset e Model?

PyTorch segue un pattern di **separazione delle responsabilità**: il `Dataset` gestisce l'accesso ai dati (qual è il campione *i*?), il `DataLoader` gestisce batching e shuffling, e il `Model` gestisce il forward pass e la loss. Questo rende ogni componente riutilizzabile e testabile indipendentemente.

### Il pattern `__getitem__`

Ogni campione restituito da `__getitem__(i)` è una tupla `(input, target)`:
- **input** — un tensore a 2 canali creato impilando il campo della previsione e il campo del lead time lungo la dimensione dei canali
- **target** — il campo verità ERA5 (1 canale)

```
forecast  (1, H, W) ─┐
                      ├─ stack ──→ input (2, H, W)
lead_time (1, H, W) ─┘

truth     (1, H, W) ──────────→ target (1, H, W)
```

### Lightning DataModule

Il `DataModule` racchiude l'intero ciclo di vita dei dati in un unico oggetto: chiama `setup()` per creare le istanze `Dataset` di train/val/test, poi espone `train_dataloader()`, `val_dataloader()` e `test_dataloader()`. Lightning li chiama automaticamente — non serve scrivere il loop boilerplate.

### Shuffling

- **Training**: con shuffle — spezza l'ordine temporale così che batch consecutivi siano decorrelati, riducendo il rumore sul gradiente e migliorando la convergenza
- **Validation / Test**: senza shuffle — garantisce metriche riproducibili e rende facile associare le predizioni a date specifiche

### Batch Size

Il training usa la **discesa del gradiente a mini-batch**: invece di calcolare la loss sull'intero dataset (lento) o su un singolo campione (rumoroso), si calcola la media su un batch di campioni. Batch più grandi producono gradienti più lisci ma richiedono più memoria GPU; batch più piccoli aggiungono rumore regolarizzante ma convergono in modo più irregolare.


In [ ]:
class PrecipitationDataset(Dataset):
    """Dataset per il post-processing della precipitazione con canale lead time."""

    def __init__(
        self,
        forecast: np.ndarray,
        truth: np.ndarray,
        lead_norm: np.ndarray,
    ) -> None:
        self.forecast = torch.from_numpy(forecast).unsqueeze(1)  # (N,1,H,W)
        self.truth = torch.from_numpy(truth).unsqueeze(1)
        self.lead_norm = torch.from_numpy(lead_norm).float()     # (N,)

    def __len__(self) -> int:
        return len(self.forecast)

    def __getitem__(
        self, idx: int
    ) -> tuple[torch.Tensor, torch.Tensor]:
        fc = self.forecast[idx]                         # (1, H, W)
        lead_ch = self.lead_norm[idx].expand_as(fc)     # (1, H, W)
        x = torch.cat([fc, lead_ch], dim=0)             # (2, H, W)
        return x, self.truth[idx]


class PrecipDataModule(pl.LightningDataModule):
    """Lightning DataModule per i dati di precipitazione."""

    def __init__(
        self,
        forecast: np.ndarray,
        truth: np.ndarray,
        lead_norm: np.ndarray,
        n_train_samples: int,
        n_trainval_samples: int,
        batch_size: int,
    ) -> None:
        super().__init__()
        self.forecast = forecast
        self.truth = truth
        self.lead_norm = lead_norm
        self.n_train = n_train_samples
        self.n_trainval = n_trainval_samples
        self.batch_size = batch_size

    def setup(self, stage: str | None = None) -> None:
        self.train_ds = PrecipitationDataset(
            self.forecast[:self.n_train],
            self.truth[:self.n_train],
            self.lead_norm[:self.n_train],
        )
        self.val_ds = PrecipitationDataset(
            self.forecast[self.n_train:self.n_trainval],
            self.truth[self.n_train:self.n_trainval],
            self.lead_norm[self.n_train:self.n_trainval],
        )
        self.test_ds = PrecipitationDataset(
            self.forecast[self.n_trainval:],
            self.truth[self.n_trainval:],
            self.lead_norm[self.n_trainval:],
        )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_ds, batch_size=self.batch_size
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self.test_ds, batch_size=self.batch_size
        )


def create_datamodule(
    data: dict[str, Any],
    cfg: DictConfig,
) -> PrecipDataModule:
    """Crea e configura il Lightning DataModule.

    Args:
        data: Dizionario restituito da ``load_data()``.
        cfg: Configurazione con chiavi ``data.*`` per split e batch.

    Returns:
        ``PrecipDataModule`` pronto all'uso (``setup()`` già chiamato).
    """
    dm = PrecipDataModule(
        data["forecast"],
        data["truth"],
        data["lead_norm"],
        n_train_samples=data["n_train_samples"],
        n_trainval_samples=data["n_trainval_samples"],
        batch_size=cfg.data.batch_size,
    )
    dm.setup()
    print(
        f"Train: {len(dm.train_ds)}, "
        f"Val: {len(dm.val_ds)}, "
        f"Test: {len(dm.test_ds)}"
    )
    return dm


dm = create_datamodule(data, cfg)


## 4. Modello: CNN semplice per il Post-Processing

Una CNN residuale che impara a correggere la previsione. L'input ha **due canali**:
1. Il campo della previsione di precipitazione
2. Un campo costante che codifica il lead time normalizzato (0 → 1)

Il modello predice un *campo di correzione* (1 canale) che viene sommato al canale della previsione tramite una connessione residuale.

### Operazione di convoluzione

Ogni strato convoluzionale applica un kernel (filtro) appreso a patch locali dell'input:

$$(f * g)(i,j) = \sum_{m,n} f(m,n) \cdot g(i{-}m,\, j{-}n)$$

Con `kernel_size=3`, il filtro è una finestra 3×3 — ogni pixel di output dipende da un piccolo intorno locale. Impostando `padding=1` le dimensioni spaziali (H, W) vengono preservate attraverso ogni strato.

### Architettura

```
Input (2, H, W) → [Conv2d → ReLU] → [Conv2d → ReLU] → Conv2d → Correzione (1, H, W)
                                                                        │
Canale forecast ───────────────────────────────────────────────── (+) ──→ Predizione
```

### Funzioni di attivazione

Tra le convoluzioni applichiamo la **ReLU** (Rectified Linear Unit):

$$\text{ReLU}(x) = \max(0, x)$$

Le attivazioni non lineari sono essenziali: senza di esse, impilare più strati lineari (convoluzionali) collasserebbe in una singola trasformazione lineare, e la rete non potrebbe apprendere pattern di correzione complessi.

### Connessione residuale

La CNN predice una **correzione** $\delta$, non l'intero campo di precipitazione. L'output finale è:

$$\hat{y} = x_{\text{forecast}} + \delta$$

Questo è molto più facile da ottimizzare: il modello parte da una previsione già ragionevole e deve solo apprendere il bias/errore sistematico. Senza la connessione residuale, la rete dovrebbe ricostruire l'intero campo spaziale da zero — un compito più difficile che spreca capacità.


In [ ]:
class PrecipPostProcessor(pl.LightningModule):
    """CNN che apprende una correzione residuale dipendente dal lead time."""

    def __init__(
        self,
        cfg: DictConfig,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] | None = None,
        activation_cls: type[nn.Module] | None = None,
    ) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=["loss_fn"])
        self.cfg = cfg
        self.loss_fn = loss_fn or nn.MSELoss()
        self.optimizer_cls = optimizer_cls or torch.optim.Adam
        activation_cls = activation_cls or nn.ReLU

        layers: list[nn.Module] = []
        in_ch: int = cfg.model.in_channels
        for i in range(cfg.model.n_layers):
            out_ch = (
                cfg.model.hidden_channels
                if i < cfg.model.n_layers - 1
                else cfg.model.out_channels
            )
            layers.append(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
            )
            if i < cfg.model.n_layers - 1:
                layers.append(activation_cls())
            in_ch = out_ch
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        correction = self.net(x)              # (B, 1, H, W)
        return x[:, :1] + correction          # residuale

    def training_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> torch.Tensor:
        x, truth = batch
        pred = self(x)
        loss = self.loss_fn(pred, truth)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> torch.Tensor:
        x, truth = batch
        pred = self(x)
        loss = self.loss_fn(pred, truth)
        mae = F.l1_loss(pred, truth)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mae", mae)
        return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        return self.optimizer_cls(
            self.parameters(), lr=self.cfg.training.learning_rate
        )


def create_model(
    cfg: DictConfig,
    model_cls: type[pl.LightningModule] = PrecipPostProcessor,
    **model_kwargs: Any,
) -> pl.LightningModule:
    """Costruisce la CNN di post-processing dalla configurazione.

    Args:
        cfg: Configurazione con chiavi ``model.*`` per l'architettura.
        model_cls: Classe LightningModule da istanziare (default:
            ``PrecipPostProcessor``).
        **model_kwargs: Argomenti keyword aggiuntivi passati a
            ``model_cls.__init__`` (es. ``loss_fn``,
            ``optimizer_cls``, ``activation_cls``).

    Returns:
        Modello non addestrato.
    """
    model = model_cls(cfg, **model_kwargs)
    print(model)
    return model


model = create_model(cfg)



## 5. Training con Lightning + Tracciamento MLFlow

Il `Trainer` di Lightning gestisce il loop di training, il posizionamento sulla GPU e il logging. Includiamo l'esecuzione in un **contesto MLFlow** così che ogni iperparametro e metrica vengano tracciati automaticamente. La funzione `train()` restituisce sia il trainer che un dizionario di metriche finali convertite in **mm** usando il fattore di scala della normalizzazione.

### Funzione di loss — MSE

Minimizziamo il **Mean Squared Error** tra predizioni e verità:

$$\mathcal{L}_{\text{MSE}} = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2$$

L'elevamento al quadrato penalizza gli errori grandi in modo sproporzionato — un errore di 10 mm contribuisce 100× più di un errore di 1 mm. Le alternative includono il **MAE** (loss L1, più robusto agli outlier) e la **Huber loss** (combina MSE per errori piccoli con MAE per errori grandi).

### Optimizer — Adam

**Adam** (Adaptive Moment Estimation) mantiene learning rate per parametro usando stime correnti della media e varianza del gradiente. Intuitivamente: adatta le dimensioni dei passi per ogni dimensione — parametri con gradienti costantemente grandi fanno passi più piccoli, e viceversa. Questo lo rende molto meno sensibile al learning rate iniziale rispetto al semplice SGD.

### Learning Rate

Il learning rate (default `1e-3`) controlla la dimensione del passo nello spazio dei parametri:
- **Troppo alto** → l'optimizer supera i minimi, la loss oscilla o diverge
- **Troppo basso** → la convergenza è estremamente lenta, il training può bloccarsi in minimi locali scadenti

La sezione Playground permette di sperimentare con diversi valori.

### Epoche

Un'**epoca** = un passaggio completo attraverso tutti i campioni di training. Con 20 epoche, il modello vede ogni campione 20 volte, affinando i suoi parametri ad ogni passaggio.

### Monitoraggio della validation

Dopo ogni epoca il trainer calcola la loss sul set di **validation** tenuto da parte. Questo rivela l'overfitting: se la loss di training continua a diminuire ma quella di validation inizia ad aumentare, il modello sta memorizzando i dati di training anziché apprendere pattern generalizzabili.


In [ ]:
def train(
    model: pl.LightningModule,
    datamodule: PrecipDataModule,
    cfg: DictConfig,
    precip_scale: float,
    run_name: str = "cnn_postprocessor",
) -> tuple[pl.Trainer, dict[str, float], str]:
    """Addestra il modello con Lightning e logga su MLFlow.

    Args:
        model: La CNN da addestrare.
        datamodule: Lightning DataModule (già configurato).
        cfg: Configurazione con chiavi ``training.*`` e ``mlflow.*``.
        precip_scale: Fattore ``p99 - p1`` per riconvertire le metriche
            normalizzate in mm.
        run_name: Nome per l'esecuzione MLflow (mostrato nella tabella).

    Returns:
        Tupla ``(trainer, metrics, run_id)`` dove *metrics* contiene
        ``val_rmse_mm`` e ``val_mae_mm``, e *run_id* è l'ID
        dell'esecuzione MLflow per loggare metriche aggiuntive.
    """
    mlflow.set_experiment(cfg.mlflow.experiment_name)

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(OmegaConf.to_container(cfg, resolve=True))

        trainer = pl.Trainer(
            max_epochs=cfg.training.max_epochs,
            accelerator=cfg.training.accelerator,
            enable_progress_bar=True,
            log_every_n_steps=5,
        )
        trainer.fit(model, datamodule)

        metrics: dict[str, float] = {}
        val_loss = trainer.callback_metrics.get("val_loss")
        val_mae = trainer.callback_metrics.get("val_mae")
        if val_loss is not None:
            metrics["val_rmse_mm"] = float(
                val_loss.item() ** 0.5 * precip_scale
            )
            mlflow.log_metric(
                "final_val_rmse_mm", metrics["val_rmse_mm"]
            )
        if val_mae is not None:
            metrics["val_mae_mm"] = float(
                val_mae.item() * precip_scale
            )
            mlflow.log_metric(
                "final_val_mae_mm", metrics["val_mae_mm"]
            )

    rmse = metrics.get("val_rmse_mm")
    mae = metrics.get("val_mae_mm")
    if rmse is not None:
        print(f"\nRMSE finale validation: {rmse:.4f} mm")
    if mae is not None:
        print(f"MAE finale validation:  {mae:.4f} mm")

    return trainer, metrics, run.info.run_id


trainer, metrics, run_id = train(model, dm, cfg, data["precip_scale"])


In [ ]:
def show_mlflow_runs(cfg: DictConfig) -> None:
    """Mostra una tabella riassuntiva delle esecuzioni MLflow per l'esperimento corrente.

    Funziona inline su Colab (non serve ``mlflow ui``).
    """
    from IPython.display import display

    experiment = mlflow.get_experiment_by_name(cfg.mlflow.experiment_name)
    if experiment is None:
        print("Nessun esperimento MLflow trovato.")
        return

    df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
    )
    if df.empty:
        print("Nessuna esecuzione registrata.")
        return

    # Costruisce una tabella riassuntiva ordinata
    cols = {}
    cols["Run Name"] = df["tags.mlflow.runName"]
    cols["Status"] = df["status"]
    cols["Start"] = pd.to_datetime(df["start_time"]).dt.strftime("%H:%M:%S")

    # Include le colonne delle metriche presenti
    for col in sorted(df.columns):
        if col.startswith("metrics."):
            short = col.replace("metrics.", "")
            cols[short] = df[col]

    # Iperparametri chiave
    for key in [
        "params.training.max_epochs",
        "params.training.learning_rate",
        "params.model.hidden_channels",
        "params.model.n_layers",
    ]:
        if key in df.columns:
            short = key.replace("params.", "")
            cols[short] = df[key]

    summary = pd.DataFrame(cols)
    display(summary)


## 6. Valutazione e Visualizzazione

La funzione `predict()` esegue l'inferenza sul test set e **denormalizza** tutti gli output riportandoli in **mm** affinché le metriche siano fisicamente interpretabili. `evaluate()` produce poi mappe di confronto e curve di skill per lead time.

### Denormalizzazione

Per riconvertire le predizioni in millimetri invertiamo lo scaling min-max:

$$x_{\text{mm}} = x_{\text{norm}} \cdot (p_{99} - p_1) + p_1$$

Riportare le metriche in unità fisiche (mm) le rende direttamente interpretabili dai previsori.

### RMSE

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2}$$

Il Root Mean Squared Error ha le **stesse unità** della variabile (mm) e penalizza fortemente gli errori grandi grazie all'elevamento al quadrato.

### MAE

$$\text{MAE} = \frac{1}{N}\sum_{i=1}^{N}|y_i - \hat{y}_i|$$

Il Mean Absolute Error applica una **penalità lineare** — è più robusto agli outlier rispetto all'RMSE e assegna uguale peso a tutte le grandezze di errore.

### Skill Score

$$\text{Skill} = 1 - \frac{\text{RMSE}_{\text{post-processed}}}{\text{RMSE}_{\text{raw}}}$$

Uno skill score positivo indica che la CNN migliora la previsione grezza; 0 = nessun miglioramento; negativo = la correzione peggiora le cose.

### Curve per lead time

L'errore di previsione tipicamente **cresce con il lead time** man mano che l'atmosfera diventa meno prevedibile. Le curve di skill mostrano se la correzione della CNN regge sugli orizzonti più lunghi o si degrada — a lead time molto lunghi la previsione grezza stessa è scadente, lasciando meno bias sistematico per la CNN da correggere.


In [ ]:
def predict(
    model: pl.LightningModule,
    datamodule: PrecipDataModule,
    data: dict[str, Any],
) -> dict[str, np.ndarray]:
    """Esegue l'inferenza sul test set e denormalizza in mm.

    Args:
        model: CNN addestrata.
        datamodule: DataModule con ``test_ds`` popolato.
        data: Dizionario restituito da ``load_data()`` (per le
            statistiche di normalizzazione e le coordinate).

    Returns:
        Dizionario con ``forecast``, ``truth``, ``pred`` (tutti in mm),
        ``lead`` (normalizzato [0, 1]), ``lats`` / ``lons``,
        ``valid_times``, ``init_times``, ``n_steps`` e
        ``step_hours``.
    """
    precip_p1 = data["precip_p1"]
    precip_scale = data["precip_scale"]

    def _to_mm(x: np.ndarray) -> np.ndarray:
        return x * precip_scale + precip_p1

    model.eval()
    test_ds = datamodule.test_ds
    test_x = torch.stack(
        [test_ds[i][0] for i in range(len(test_ds))]
    )

    with torch.no_grad():
        test_pred = model(test_x)

    # Usa direttamente gli init time fissi del test
    test_init_times = data["test_init_times"]
    n_steps = data["n_steps"]
    step_hours = data["step_hours"]

    # Costruisce valid time e init time per campione
    step_deltas = (step_hours * 3600).astype("timedelta64[s]")
    valid_times = np.repeat(test_init_times, n_steps) + np.tile(
        step_deltas, len(test_init_times)
    )
    sample_init_times = np.repeat(test_init_times, n_steps)

    return {
        "forecast": _to_mm(test_x[:, 0].numpy()),
        "truth": _to_mm(test_ds.truth.squeeze(1).numpy()),
        "pred": _to_mm(test_pred.squeeze(1).numpy()),
        "lead": test_ds.lead_norm.numpy(),
        "lats": data["lats"],
        "lons": data["lons"],
        "valid_times": valid_times,
        "init_times": sample_init_times,
        "n_steps": n_steps,
        "step_hours": step_hours,
    }


predictions = predict(model, dm, data)


In [ ]:
def evaluate(
    predictions: dict[str, np.ndarray],
) -> dict[str, float]:
    """Genera mappe di errore e curve RMSE / MAE per lead time.

    Args:
        predictions: Dizionario restituito da ``predict()`` con chiavi
            ``forecast``, ``truth``, ``pred``, ``lead``, ``lats``,
            ``lons``, ``init_times``, ``n_steps``, ``step_hours``
            (tutti in mm tranne ``lead``).

    Returns:
        Dizionario di metriche di valutazione (tutte in mm o percentuale).
    """
    fc = predictions["forecast"]
    tr = predictions["truth"]
    pp = predictions["pred"]
    lead = predictions["lead"]
    lats = predictions["lats"]
    lons = predictions["lons"]
    init_times = predictions["init_times"]
    n_steps = predictions["n_steps"]
    step_hours = predictions["step_hours"]
    max_hours = float(step_hours[-1])
    lon2d, lat2d = np.meshgrid(lons, lats)

    # --- Mappe di errore (2 righe × 2 colonne) ---
    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        2, 2, figsize=(14, 10), subplot_kw={"projection": proj}
    )

    for row, idx in enumerate([0, n_steps - 1]):
        lead_h = step_hours[idx % n_steps]
        init_dt = pd.Timestamp(init_times[idx])
        init_str = init_dt.strftime("%Y-%m-%d %H:%M")

        hres_err = fc[idx] - tr[idx]
        pp_err = pp[idx] - tr[idx]

        # Limiti di colore simmetrici
        abs_max = max(
            np.abs(hres_err).max(), np.abs(pp_err).max()
        )

        panels = [
            (f"HRES \u2212 Verit\u00e0 | {init_str} +{lead_h:.0f}h",
             hres_err),
            (f"CNN \u2212 Verit\u00e0 | {init_str} +{lead_h:.0f}h",
             pp_err),
        ]
        for col, (title, err) in enumerate(panels):
            ax = axes[row, col]
            im = ax.pcolormesh(
                lon2d, lat2d, err, cmap="RdBu_r",
                vmin=-abs_max, vmax=abs_max, transform=proj,
            )
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
            ax.add_feature(
                cfeature.BORDERS, linewidth=0.3, linestyle="--"
            )
            ax.gridlines(
                draw_labels=(row == 1), linewidth=0.3, alpha=0.5
            )
            ax.set_title(title)
            plt.colorbar(
                im, ax=ax, orientation="horizontal",
                pad=0.05, shrink=0.8,
            )

            # Annotazione RMSE e MAE per questo pannello
            rmse = np.sqrt(np.mean(err ** 2))
            mae = np.mean(np.abs(err))
            ax.text(
                0.02, 0.05,
                f"RMSE={rmse:.3f}  MAE={mae:.3f}",
                transform=ax.transAxes,
                fontsize=8,
                va="bottom",
                ha="left",
                bbox=dict(
                    facecolor="white", alpha=0.7, edgecolor="none"
                ),
            )

    plt.suptitle(
        "Mappe di errore della previsione \u2014 Test set (mm)",
        fontsize=14, y=1.02,
    )
    plt.tight_layout()
    plt.show()

    # --- Curve di skill per lead time ---
    unique_leads = np.unique(lead)
    lead_hours = unique_leads * max_hours
    raw_rmses, pp_rmses = [], []
    raw_maes, pp_maes = [], []

    for lv in unique_leads:
        mask = lead == lv
        raw_rmses.append(
            np.sqrt(np.mean((fc[mask] - tr[mask]) ** 2))
        )
        pp_rmses.append(
            np.sqrt(np.mean((pp[mask] - tr[mask]) ** 2))
        )
        raw_maes.append(np.mean(np.abs(fc[mask] - tr[mask])))
        pp_maes.append(np.mean(np.abs(pp[mask] - tr[mask])))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(lead_hours, raw_rmses, "o-", label="HRES grezzo")
    ax1.plot(lead_hours, pp_rmses, "s-", label="Post-processato (CNN)")
    ax1.set_xlabel("Lead time (ore)")
    ax1.set_ylabel("RMSE (mm)")
    ax1.set_title("RMSE per lead time (test set)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(lead_hours, raw_maes, "o-", label="HRES grezzo")
    ax2.plot(lead_hours, pp_maes, "s-", label="Post-processato (CNN)")
    ax2.set_xlabel("Lead time (ore)")
    ax2.set_ylabel("MAE (mm)")
    ax2.set_title("MAE per lead time (test set)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # --- Risultati complessivi ---
    raw_rmse = np.sqrt(np.mean((fc - tr) ** 2))
    pp_rmse = np.sqrt(np.mean((pp - tr) ** 2))
    raw_mae = np.mean(np.abs(fc - tr))
    pp_mae = np.mean(np.abs(pp - tr))
    print(f"RMSE previsione HRES grezza: {raw_rmse:.4f} mm")
    print(f"RMSE post-processato:        {pp_rmse:.4f} mm")
    print(
        f"Miglioramento RMSE:          "
        f"{(1 - pp_rmse / raw_rmse) * 100:.1f}%"
    )
    print(f"\nMAE previsione HRES grezza:  {raw_mae:.4f} mm")
    print(f"MAE post-processato:         {pp_mae:.4f} mm")
    print(
        f"Miglioramento MAE:           "
        f"{(1 - pp_mae / raw_mae) * 100:.1f}%"
    )

    eval_metrics = {
        "eval_raw_hres_rmse_mm": float(raw_rmse),
        "eval_raw_hres_mae_mm": float(raw_mae),
        "eval_pp_rmse_mm": float(pp_rmse),
        "eval_pp_mae_mm": float(pp_mae),
        "eval_rmse_improvement_pct": float(
            (1 - pp_rmse / raw_rmse) * 100
        ),
        "eval_mae_improvement_pct": float(
            (1 - pp_mae / raw_mae) * 100
        ),
    }
    return eval_metrics


eval_metrics = evaluate(predictions)

# Logga le metriche di valutazione nella stessa esecuzione MLflow
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics(eval_metrics)


## 7. Playground

Esegui l'intera pipeline in una sola cella. **Modifica i parametri qui sotto** e riesegui per sperimentare!

Puoi personalizzare:
- **Valori di configurazione** — `max_lead_time`, `max_epochs`, `hidden_channels`, `n_layers`, `learning_rate`, `batch_size`
- **Intervallo temporale** — `time_start` / `time_end` (dati HRES disponibili dal `2016-01-01` al `2022-12-31`)
- **Funzione di loss** — sostituisci `nn.MSELoss()` con `nn.L1Loss()`, `nn.HuberLoss()`, ecc.
- **Optimizer** — prova `torch.optim.AdamW`, `torch.optim.SGD`, ecc.
- **Attivazione** — prova `nn.GELU`, `nn.SiLU`, `nn.LeakyReLU`, ecc.
- **Classe del modello** — definisci il tuo `LightningModule` e passalo tramite `model_cls`


In [ ]:
# ── 1. Configurazione (modifica questi!) ──────────────────────
# time_start / time_end: periodo train+val (HRES disponibile 2016-01-01 — 2022-12-31)
# train_split: frazione degli init time train+val usata per il training (il resto → val)
# test_start / test_end: periodo di test (ERA5 disponibile fino al 2021-12-31, quindi
#   test_end deve essere almeno max_lead_time prima di tale data)
run_name = "cnn_postprocessor"  # cambia questo prima di ogni esecuzione!

cfg = load_config(
    "../configs/precipitation.yaml",
    time_start="2018-01-01",
    time_end="2018-02-28",
    test_start="2021-01-01",    # periodo di test fisso (ERA5 disponibile fino al 2021-12-31)
    test_end="2021-02-28",
    train_split=0.6,
    max_lead_time="2 days",
    max_epochs=20,
    hidden_channels=16,
    n_layers=3,
    learning_rate=1e-3,
    batch_size=32,
)

# ── 2. Funzione di loss ────────────────────────────────────────
# Prova: nn.L1Loss(), nn.HuberLoss(), nn.SmoothL1Loss()
loss_fn = nn.MSELoss()

# ── 3. Optimizer ───────────────────────────────────────────────
# Prova: torch.optim.SGD, torch.optim.AdamW
optimizer_cls = torch.optim.Adam

# ── 4. Attivazione ─────────────────────────────────────────────
# Prova: nn.GELU, nn.SiLU, nn.LeakyReLU
activation_cls = nn.ReLU

# ── 5. Esecuzione pipeline ─────────────────────────────────────
data = load_data(cfg)
plot_raw_data(data)
dm = create_datamodule(data, cfg)
model = create_model(
    cfg,
    loss_fn=loss_fn,
    optimizer_cls=optimizer_cls,
    activation_cls=activation_cls,
)
trainer, metrics, run_id = train(
    model, dm, cfg, data["precip_scale"], run_name=run_name,
)
predictions = predict(model, dm, data)
eval_metrics = evaluate(predictions)

# Logga le metriche di valutazione nella stessa esecuzione MLflow
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics(eval_metrics)

# ── 6. Confronta esperimenti ──────────────────────────────────
show_mlflow_runs(cfg)


## 8. Esercizi

Usa la cella **Playground** qui sopra per sperimentare — cambia un parametro alla volta e riesegui.

1. **Aumenta l'orizzonte**: Imposta `max_lead_time="5 days"` o `"10 days"` — come cambia la curva di skill per lead time?
2. **Modifica l'architettura**: Prova `n_layers=5` o `hidden_channels=32`. Cosa succede con un modello più profondo / più largo?
3. **Confronta gli optimizer**: Imposta `optimizer_cls = torch.optim.SGD` o `torch.optim.AdamW` nel playground — come cambia la convergenza?
4. **Aggiungi più canali di input**: Carica variabili HRES aggiuntive dallo store Zarr (es. `2m_temperature`, `geopotential`) come canali extra per la CNN
5. **Prova una regione diversa**: Cambia `lat_slice` per selezionare diverse bande di latitudine e osserva come cambiano i pattern di precipitazione
6. **Confronta esperimenti**: Chiama `show_mlflow_runs(cfg)` per vedere un riepilogo di tutte le esecuzioni affiancate. Per l'interfaccia completa, gli utenti locali possono eseguire `!mlflow ui` e aprire `http://localhost:5000`
